# Lab 08: Greedy Algorithms

**EECS 291 — Data Structures & Algorithms**

This lab notebook covers the key greedy algorithm patterns from Lecture 16.
After working through these examples, you'll implement them yourself in PrairieLearn.

## Agenda

1. Quick recap: what is the greedy pattern?
2. **Task Deadline Scheduling** — step-by-step walkthrough
3. **Fractional vs 0/1 Knapsack** — when greedy works and when it doesn't
4. **Coin Change** — greedy works for US coins (fails for arbitrary denominations)
5. **Weighted Activity Selection** — when greedy is suboptimal


## The Greedy Pattern

A **greedy algorithm** makes a locally optimal choice at each step,
hoping it leads to a globally optimal solution.

**Template:**
```
1. Sort or organize the options
2. Pick the best available option
3. Update what's still available
4. Repeat until done
```

**When does greedy work?**
- **Greedy choice property**: local optimal → global optimal
- **Optimal substructure**: optimal solution contains optimal sub-solutions

**When does greedy fail?**
- When a locally good choice blocks a much better future choice

## Part 1: Task Deadline Scheduling

**Problem:** You're a tour manager scheduling sound checks before the big show.
Each artist has a **deadline** (latest time slot they can use) and a **penalty fee** if they miss their sound check.
Each sound check takes exactly 1 time unit (slots are 1, 2, 3, …).

**Key insight:** To minimize penalties, always schedule the **most expensive** task first —
that way we protect the costliest penalties from being incurred.


In [ ]:
# Sound checks: (deadline, penalty, artist)
sound_checks = [
    (1, 5,  'The Algorithm'),
    (2, 10, 'Heap Sort'),
    (2, 3,  'Binary Trees'),
]

print('Sound checks (deadline, penalty, artist):')
for deadline, penalty, artist in sound_checks:
    print(f'  {artist}: deadline=slot {deadline}, penalty=${penalty}')


In [ ]:
# Step 1: Sort by PENALTY descending (protect most expensive tasks first)
sorted_checks = sorted(sound_checks, key=lambda x: x[1], reverse=True)

print('Sorted by penalty (descending):')
for deadline, penalty, artist in sorted_checks:
    print(f'  {artist}: deadline=slot {deadline}, penalty=${penalty}')


In [ ]:
# Step 2: Greedily assign each task to latest available slot ≤ deadline
occupied = set()
total_penalty = 0

for deadline, penalty, artist in sorted_checks:
    # Find the latest available slot at or before the deadline
    scheduled = False
    for slot in range(deadline, 0, -1):
        if slot not in occupied:
            occupied.add(slot)
            scheduled = True
            print(f'✅ Scheduled: {artist} in slot {slot} (deadline={deadline})')
            break
    if not scheduled:
        total_penalty += penalty
        print(f'❌ Late: {artist} — no slot ≤ {deadline} available, penalty=${penalty}')

print(f'\nTotal penalty: ${total_penalty}')


In [ ]:
# The greedy function
def min_penalty(tasks: list[tuple[int, int]]) -> int:
    """Return minimum total penalty for late tasks."""
    if not tasks:
        return 0
    sorted_tasks = sorted(tasks, key=lambda t: t[1], reverse=True)
    occupied = set()
    total_penalty = 0
    for deadline, penalty in sorted_tasks:
        scheduled = False
        for slot in range(deadline, 0, -1):
            if slot not in occupied:
                occupied.add(slot)
                scheduled = True
                break
        if not scheduled:
            total_penalty += penalty
    return total_penalty

# Test it
tasks_only = [(d, p) for d, p, _ in sound_checks]
print(f'min_penalty(example): {min_penalty(tasks_only)}')   # → 3

# Another example: 3 tasks all with deadline 1 → only one fits
print(f'min_penalty([(1,10),(1,5),(1,1)]): {min_penalty([(1,10),(1,5),(1,1)])}')  # → 6


## Part 2: Fractional Knapsack

**Problem:** Loading a tour bus with equipment. Each item has a weight and value.
You can take a fraction of any item.

**Key insight:** Always take from the item with the highest **value per kg**.
Splitting is allowed, so you never 'waste' capacity.

| Item | Weight (kg) | Value ($) | $/kg |
|------|-------------|-----------|------|
| Guitar amp | 20 | 80 | 4.0 |
| PA system | 30 | 150 | 5.0 |
| Cables | 10 | 30 | 3.0 |
| Lighting | 15 | 45 | 3.0 |

In [ ]:
# Tour bus equipment: (weight_kg, value_dollars, name)
equipment = [
    (20, 80, 'Guitar amp'),
    (30, 150, 'PA system'),
    (10, 30, 'Cables'),
    (15, 45, 'Lighting'),
]
bus_capacity = 40  # kg

# Sort by value/weight ratio
sorted_eq = sorted(equipment, key=lambda x: x[1]/x[0], reverse=True)
print('Sorted by $/kg (highest first):')
for w, v, name in sorted_eq:
    print(f'  {name}: {v/w:.2f} $/kg ({w}kg, ${v})')

In [ ]:
# Greedy fractional knapsack
total_value = 0.0
remaining = bus_capacity

print(f'Bus capacity: {bus_capacity}kg\n')
for weight, value, name in sorted_eq:
    if remaining <= 0:
        break
    if weight <= remaining:
        total_value += value
        remaining -= weight
        print(f'✅ Take all of {name}: {weight}kg, ${value} | remaining: {remaining}kg')
    else:
        fraction = remaining / weight
        taken_value = value * fraction
        total_value += taken_value
        print(f'✅ Take {fraction:.0%} of {name}: {remaining}kg, ${taken_value:.1f} | remaining: 0kg')
        remaining = 0

print(f'\nTotal value loaded: ${total_value:.1f}')

### Why does fractional greedy work?

Because we can always take exactly as much of the best item as we have room for.
There's no 'wasted capacity' — fractions fill the gap perfectly.

**0/1 Knapsack** (no fractions allowed) requires dynamic programming.
Greedy fails there because taking the best-ratio item might not leave room
for a combination of smaller items with higher total value.

In [ ]:
# Counter-example: greedy FAILS for 0/1 knapsack
# Items: (weight, value)
items_01 = [(10, 60), (20, 100), (30, 120)]
capacity_01 = 50

# Greedy by ratio: 6.0, 5.0, 4.0 → take item 0 (10kg, 60), item 1 (20kg, 100)
# That's 30kg, $160 — can't take item 2 (30kg) without splitting
greedy_01 = 60 + 100  # $160

# True optimal (0/1): take item 1 (20kg) + item 2 (30kg) = 50kg, $220
optimal_01 = 100 + 120  # $220

print(f'Greedy 0/1 result: ${greedy_01}')
print(f'True optimal (DP): ${optimal_01}')
print(f'Greedy misses ${optimal_01 - greedy_01} in value!')

## Part 3: Greedy Coin Change

**Problem:** Merch booth needs to make change. Minimum coins to reach the amount.

**Greedy:** Always use the largest coin that fits.

This works for US coins (25¢, 10¢, 5¢, 1¢) because each denomination
is a 'nice' multiple of the smaller ones.

But it **fails** for some other denomination sets!

In [ ]:
# US coins: greedy works
def make_change_greedy(coins: list[int], amount: int) -> tuple[int, list]:
    """Return (count, breakdown) of coins used."""
    sorted_coins = sorted(coins, reverse=True)
    total = 0
    used = []
    remaining = amount
    for coin in sorted_coins:
        count = remaining // coin
        if count > 0:
            total += count
            used.append((coin, count))
            remaining -= count * coin
    return total, used

us_coins = [25, 10, 5, 1]
amount = 41
count, breakdown = make_change_greedy(us_coins, amount)
print(f'Making change for {amount}¢:')
for coin, n in breakdown:
    print(f'  {n} × {coin}¢')
print(f'Total coins: {count}')

In [ ]:
# Counter-example: coins [1, 3, 4], amount = 6
# Greedy: 4 + 1 + 1 = 3 coins
# Optimal: 3 + 3 = 2 coins

bad_coins = [1, 3, 4]
count_greedy, breakdown_greedy = make_change_greedy(bad_coins, 6)
print(f'Greedy for 6¢ with coins {bad_coins}:')
for coin, n in breakdown_greedy:
    print(f'  {n} × {coin}¢')
print(f'Greedy: {count_greedy} coins')
print(f'Optimal: 2 coins (3¢ + 3¢)')
print()
print('US coins work because each denomination is designed to be greedy-compatible.')
print('Arbitrary denominations may not have this property.')

## Part 4: Weighted Activity Selection — Where Greedy Fails

**Problem:** Same as interval scheduling, but each show earns different revenue.
We want to **maximize total revenue**, not just count of shows.

**Greedy by end time** doesn't consider value — it may skip a lucrative
long show in favor of two cheap short ones.

In [ ]:
# Weighted activities: (start, end, revenue)
shows = [
    (1, 5, 10, 'Local Band A'),
    (6, 10, 10, 'Local Band B'),
    (1, 10, 100, 'Headliner'),
]

print('Shows:')
for s, e, v, name in shows:
    print(f'  {name}: {s}:00-{e}:00, ${v}')

In [ ]:
# Greedy approach: sort by end time, pick non-conflicting
sorted_shows = sorted(shows, key=lambda x: x[1])
selected = []
last_end = 0
total_value = 0

for start, end, value, name in sorted_shows:
    if start >= last_end:
        selected.append(name)
        total_value += value
        last_end = end
        print(f'✅ Pick {name}: +${value}')
    else:
        print(f'❌ Skip {name}: conflicts')

print(f'\nGreedy total: ${total_value}')
print(f'Optimal (just Headliner): $100')
print(f'\nGreedy missed ${100 - total_value} in revenue!')

### Key Takeaway

| Problem | Greedy? | Strategy |
|---------|---------|----------|
| Task deadline scheduling | ✅ Optimal | Sort by penalty desc, latest slot ≤ deadline |
| Fractional knapsack | ✅ Optimal | Sort by value/weight |
| US coin change | ✅ Optimal | Largest coin first |
| Weighted activity selection | ❌ Suboptimal | Needs DP |
| 0/1 Knapsack | ❌ Suboptimal | Needs DP |

Greedy works when making the locally best choice never prevents a globally better outcome.


## Lab 08 — Your Tasks

Open PrairieLearn and complete all 4 questions:

1. **l8-01**: `min_penalty(tasks)` — task deadline scheduling (sort by penalty desc)
2. **l8-02**: `min_merge_cost(playlists)` — minimum cost to merge playlists
3. **l8-03**: `make_change(coins, amount)` — minimum coins (greedy-friendly denominations)
4. **l8-04**: `greedy_activity_value(activities)` — weighted greedy (observe suboptimality)

**Tips:**
- All four use the same basic pattern: sort, then greedily pick
- Read the docstrings carefully — they spell out the exact strategy
- For l8-04, implement greedy-by-end-time even though it's not always optimal

Good luck! 🎸
